# TaylorMade product-image puller

Open the TaylorMade product or category page in your normal browser, then run this in its DevTools Console. It copies every loaded _D through _D5 image URL to your clipboard:

    const imageUrls = [
      ...new Set(
        performance.getEntriesByType("resource")
          .map(entry => entry.name)
          .filter(url => /_(D|D2|D3|D4|D5)\.(jpg|jpeg|png|webp)(\?|$)/i.test(url))
      )
    ];

    console.log(imageUrls.join("\n"));
    copy(imageUrls.join("\n"));

Paste the copied URL list into IMAGE_URLS_TEXT in the next cell. It validates that list and downloads the images.

In [ ]:
from pathlib import Path
from urllib.parse import urlparse
import re

import requests


PROJECT_ROOT = next(
    parent
    for parent in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
    if (parent / "requirements.txt").is_file()
)
OUTPUT_DIR = PROJECT_ROOT / "assets" / "club-reference-images" / "wood" / "driver"
IMAGE_URL_PATTERN = re.compile(
    r"_(?:D|D2|D3|D4|D5)\.(?:jpg|jpeg|png|webp)(?:\?.*)?$", re.IGNORECASE
)


# Paste the console output here: one URL per line.
IMAGE_URLS_TEXT = """
"""

IMAGE_URLS = sorted(
    {
        url
        for line in IMAGE_URLS_TEXT.splitlines()
        if (url := line.strip()) and IMAGE_URL_PATTERN.search(url)
    }
)
if not IMAGE_URLS:
    raise RuntimeError(
        "No matching image URLs were found. Paste the console output into IMAGE_URLS_TEXT, then run this cell again."
    )

print(f"Found {len(IMAGE_URLS)} matching image URLs.")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

session = requests.Session()
session.headers.update(
    {
        "User-Agent": (
            "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
            "AppleWebKit/537.36 Chrome/142.0 Safari/537.36"
        ),
        "Referer": "https://www.taylormadegolf.com/",
    }
)

for index, image_url in enumerate(IMAGE_URLS, start=1):
    filename = Path(urlparse(image_url).path).name or f"image_{index}.jpg"
    destination = OUTPUT_DIR / filename

    try:
        response = session.get(image_url, timeout=30)
        response.raise_for_status()
        destination.write_bytes(response.content)
        print(f"Downloaded: {destination}")
    except requests.RequestException as error:
        print(f"Failed: {image_url}\nReason: {error}")
